# Data Preprocessing

### Imports

In [12]:
import sys
import os
import pandas as pd

### Load Dataset

In [13]:
# Allow the dataset to be loaded with both a Google Colab kernel and a local kernel

# Load the dataset with Google Colab kernel and Drive file
if 'google.colab' in sys.modules:
    from pydrive2.auth import GoogleAuth
    from pydrive2.drive import GoogleDrive
    from google.colab import auth # type: ignore
    from oauth2client.client import GoogleCredentials

    # Authenticate the User in Google Drive
    auth.authenticate_user()
    gauth = GoogleAuth()
    gauth.credentials = GoogleCredentials.get_application_default()
    drive = GoogleDrive(gauth)

    # Google Drive ID for public sharing of the dataset
    file_id = "17FCZ63-6OyiagZWTmDPYtGajyJFzhtmd"
    file = drive.CreateFile({'id': file_id})
    file.GetContentFile('raw_data.csv')

    # Reading the csv and loading it into a pandas dataframe (Use pyarrow to prevent OOM error when loading)
    df = pd.read_csv("raw_data.csv", engine="pyarrow", dtype_backend="pyarrow")

# Load the dataset with local kernel and local file
else:
    # Reading the csv and loading it into a pandas dataframe (Use pyarrow to prevent OOM error when loading)
    df = pd.read_csv("../data/raw/raw_data.csv", engine="pyarrow", dtype_backend="pyarrow")

# Stop Jupyter Notebook from limiting the output
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Display dataset head
df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.8,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.8,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


### Delete & Transform Columns & Rows

Display the missing values per column:

In [14]:
def print_column_info(df):
    # Get the total number of rows in the DataFrame
    total_rows = len(df)

    # Create the summary DataFrame
    data = pd.DataFrame({
        'Total (Null/Total)': df.isnull().sum().astype(str) + '/' + str(total_rows),
        'Percentage (%)': df.isnull().mean() * 100,
        'Data Type': df.dtypes
    })

    # Display the result
    display(data)

print_column_info(df)

,Total (Null/Total),Percentage (%),Data Type
instant,0/17379,0.0,int64[pyarrow]
dteday,0/17379,0.0,date32[day][pyarrow]
season,0/17379,0.0,int64[pyarrow]
yr,0/17379,0.0,int64[pyarrow]
mnth,0/17379,0.0,int64[pyarrow]
hr,0/17379,0.0,int64[pyarrow]
holiday,0/17379,0.0,int64[pyarrow]
weekday,0/17379,0.0,int64[pyarrow]
workingday,0/17379,0.0,int64[pyarrow]
weathersit,0/17379,0.0,int64[pyarrow]


Modify Rows:

In [15]:
# All the rows are correct, no need to modify

Modify Columns:

In [16]:
# instant: Not Modified

# dteday: Divide the column in 3, one for year, one for month and one for day, remove the original column, and Change Data Type to Integer

df['year'] = pd.to_datetime(df['dteday'], format='%Y-%m-%d').dt.year
df['year'] = df['year'].astype('int64[pyarrow]')

df['month'] = pd.to_datetime(df['dteday'], format='%Y-%m-%d').dt.month
df['month'] = df['month'].astype('int64[pyarrow]')

df['day'] = pd.to_datetime(df['dteday'], format='%Y-%m-%d').dt.day
df['day'] = df['day'].astype('int64[pyarrow]')

df = df.drop(columns=['dteday'])

# season: Not Modified

# yr: Delete because it is the same as/extremely similar to year
df = df.drop(columns=["yr"])

# mnth: Delete because it is the same as/extremely similar to mnth
df = df.drop(columns=["mnth"])

# hr: Change name to maintain naming consistency
df.rename(columns={'hr': 'hour'}, inplace=True)

# holiday: Change Data Type to Bool
df['holiday'] = df['holiday'].astype('bool[pyarrow]')

# weekday: Not Modified

# workingday: Change Data Type to Bool
df['workingday'] = df['workingday'].astype('bool[pyarrow]')

# weathersit: Not Modified

# temp: Not Modified

# atemp: Not Modified

# hum: Not Modified

# windspeed: Not Modified

# casual: Not Modified

# registered: Not Modified

# cnt: Not Modified

# datetime: Create column to use all the time information, this is used as it is a Time Series analysis
df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])


print_column_info(df)
df.head()

,Total (Null/Total),Percentage (%),Data Type
instant,0/17379,0.0,int64[pyarrow]
season,0/17379,0.0,int64[pyarrow]
hour,0/17379,0.0,int64[pyarrow]
holiday,0/17379,0.0,bool[pyarrow]
weekday,0/17379,0.0,int64[pyarrow]
workingday,0/17379,0.0,bool[pyarrow]
weathersit,0/17379,0.0,int64[pyarrow]
temp,0/17379,0.0,double[pyarrow]
atemp,0/17379,0.0,double[pyarrow]
hum,0/17379,0.0,double[pyarrow]


,instant,season,hour,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,year,month,day,datetime
0,1,1,0,False,6,False,1,0.24,0.2879,0.81,0.0,3,13,16,2011,1,1,2011-01-01 00:00:00
1,2,1,1,False,6,False,1,0.22,0.2727,0.8,0.0,8,32,40,2011,1,1,2011-01-01 01:00:00
2,3,1,2,False,6,False,1,0.22,0.2727,0.8,0.0,5,27,32,2011,1,1,2011-01-01 02:00:00
3,4,1,3,False,6,False,1,0.24,0.2879,0.75,0.0,3,10,13,2011,1,1,2011-01-01 03:00:00
4,5,1,4,False,6,False,1,0.24,0.2879,0.75,0.0,0,1,1,2011,1,1,2011-01-01 04:00:00


### Convert to Processed .csv

In [17]:
if 'google.colab' in sys.modules:
    pass

else:
    if os.path.exists("../data/processed/processed_data.csv"):
        print("Dataset already exists")

    else:
        df.to_csv("../data/processed/processed_data.csv", index=False)
        print("Dataset saved successfully")

Dataset saved successfully
